# Career Development Index — Query Tool

Interactive utility notebook for querying the Global Career Development Index data.

Run each cell to load data and define query functions, then use them in new cells.

**Note:** This notebook is designed to be executed interactively.

In [ ]:
import pandas as pd
from pathlib import Path

def load_all_data():
    """Load all 12 CSV files and return concatenated DataFrame."""
    csv_dir = Path("../data/csv")
    if not csv_dir.exists():
        csv_dir = Path("data/csv")  # fallback for running from root
    dfs = []
    for f in sorted(csv_dir.glob("*.csv")):
        dfs.append(pd.read_csv(f))
    df = pd.concat(dfs, ignore_index=True)
    print(f"Loaded {len(df)} records, {df['sub_category'].nunique()} occupations, "
          f"{df['country_or_region'].nunique()} countries/regions")
    return df

df = load_all_data()


In [ ]:
def find_jobs(country=None, category=None, keyword=None,
              min_composite=None, min_ai_resistance=None,
              min_remote=None, min_salary=None, sort_by="composite_index",
              top_n=20):
    """Find jobs matching given filters.

    Args:
        country: Country name (Chinese) or iso_code, e.g. '中国' or 'CN'
        category: major_category or major_code, e.g. 'TECH'
        keyword: Search in sub_category or sub_category_en
        min_composite: Minimum composite_index
        min_ai_resistance: Minimum ai_resistance score
        min_remote: Minimum remote_friendly score
        min_salary: Minimum value_added score
        sort_by: Column to sort by (default: composite_index)
        top_n: Number of results to show (default: 20)

    Returns:
        Filtered and sorted DataFrame
    """
    result = df.copy()

    if country:
        mask = (result["country_or_region"] == country) | (result["iso_code"] == country)
        result = result[mask]

    if category:
        mask = (result["major_category"] == category) | (result["major_code"] == category)
        result = result[mask]

    if keyword:
        mask = (result["sub_category"].str.contains(keyword, case=False, na=False) |
                result["sub_category_en"].str.contains(keyword, case=False, na=False))
        result = result[mask]

    if min_composite is not None:
        result = result[result["composite_index"] >= min_composite]
    if min_ai_resistance is not None:
        result = result[result["ai_resistance"] >= min_ai_resistance]
    if min_remote is not None:
        result = result[result["remote_friendly"] >= min_remote]
    if min_salary is not None:
        result = result[result["value_added"] >= min_salary]

    result = result.sort_values(sort_by, ascending=False).head(top_n)

    display_cols = [
        "sub_category", "sub_category_en", "country_or_region",
        "major_code", "composite_index", "ai_resistance",
        "value_added", "cost_performance", "remote_friendly",
        "growth_coeff", "trend_5yr",
    ]
    return result[[c for c in display_cols if c in result.columns]]

# Example: find_jobs(country="CN", min_ai_resistance=7, min_remote=7)


In [ ]:
def compare(occupation, countries=None):
    """Compare an occupation across multiple countries.

    Args:
        occupation: sub_category name (Chinese), e.g. '前端工程师'
        countries: List of country names or iso_codes. If None, show all.

    Returns:
        Comparison DataFrame
    """
    result = df[df["sub_category"] == occupation].copy()
    if countries:
        mask = result["country_or_region"].isin(countries) | result["iso_code"].isin(countries)
        result = result[mask]

    result = result.sort_values("composite_index", ascending=False)

    display_cols = [
        "country_or_region", "iso_code", "composite_index",
        "value_added", "cost_performance", "ai_resistance",
        "growth_coeff", "supply_demand", "developed_scarcity",
        "remote_friendly", "stability", "trend_2000_2026", "trend_5yr",
    ]
    return result[[c for c in display_cols if c in result.columns]]

# Example: compare("前端工程师", ["CN", "US", "JP", "DE", "IN"])


In [ ]:
def top_n(category=None, metric="composite_index", n=20, country=None):
    """Get top N occupations by a given metric.

    Args:
        category: major_code or major_category (None = all)
        metric: Score column to rank by
        n: Number of results
        country: Optional country filter

    Returns:
        Top N DataFrame
    """
    result = df.copy()
    if category:
        mask = (result["major_category"] == category) | (result["major_code"] == category)
        result = result[mask]
    if country:
        mask = (result["country_or_region"] == country) | (result["iso_code"] == country)
        result = result[mask]

    result = result.nlargest(n, metric)

    display_cols = [
        "sub_category", "sub_category_en", "country_or_region",
        "major_code", metric, "composite_index",
        "value_added", "ai_resistance", "growth_coeff", "trend_5yr",
    ]
    # Deduplicate while preserving order
    seen = set()
    unique_cols = []
    for c in display_cols:
        if c not in seen and c in result.columns:
            seen.add(c)
            unique_cols.append(c)
    return result[unique_cols]

# Example: top_n(category="TECH", metric="ai_resistance", n=15, country="US")


In [ ]:
def country_overview(iso_code):
    """Get an overview of a country's career landscape.

    Args:
        iso_code: Country ISO code (e.g. 'CN', 'US', 'JP')

    Returns:
        Summary statistics by major_category
    """
    result = df[df["iso_code"] == iso_code].copy()
    country_name = result["country_or_region"].iloc[0] if len(result) > 0 else iso_code

    print(f"=== {country_name} ({iso_code}) ===")
    print(f"Total occupations: {result['sub_category'].nunique()}")
    print(f"Total records: {len(result)}")
    print(f"Mean composite_index: {result['composite_index'].mean():.2f}")
    print(f"Mean ai_resistance: {result['ai_resistance'].mean():.2f}")
    print(f"Mean trend_2000_2026: {result['trend_2000_2026'].mean():.2f}")
    print()

    summary = result.groupby("major_category").agg(
        n=("sub_category", "nunique"),
        mean_composite=("composite_index", "mean"),
        mean_ai=("ai_resistance", "mean"),
        mean_growth=("growth_coeff", "mean"),
        mean_value=("value_added", "mean"),
        mean_trend=("trend_2000_2026", "mean"),
    ).round(2).sort_values("mean_composite", ascending=False)
    return summary

# Example: country_overview("CN")


In [ ]:
def transition_path(from_occupation, target_category=None, country=None, top_n=10):
    """Suggest career transition paths from a given occupation.

    Finds occupations with high skill_versatility and career_switch scores
    in the target category, comparing key metrics.

    Args:
        from_occupation: Current sub_category name (Chinese)
        target_category: Target major_code (e.g. 'TECH'). None = all.
        country: Optional country filter (iso_code)
        top_n: Number of suggestions

    Returns:
        DataFrame with transition suggestions and score comparisons
    """
    # Get source occupation info
    source = df[df["sub_category"] == from_occupation]
    if country:
        source = source[source["iso_code"] == country]
    if len(source) == 0:
        print(f"Occupation '{from_occupation}' not found.")
        return pd.DataFrame()
    source_row = source.iloc[0]

    # Find target occupations
    targets = df.copy()
    if target_category:
        mask = (targets["major_category"] == target_category) | (targets["major_code"] == target_category)
        targets = targets[mask]
    if country:
        targets = targets[targets["iso_code"] == country]

    # Score transition viability
    targets = targets.copy()
    targets["transition_score"] = (
        targets["skill_versatility"] * 0.3 +
        targets["career_switch"] * 0.3 +
        targets["growth_coeff"] * 0.2 +
        targets["composite_index"] * 0.2
    ).round(2)

    targets = targets.nlargest(top_n, "transition_score")

    display_cols = [
        "sub_category", "sub_category_en", "country_or_region",
        "major_code", "transition_score", "skill_versatility",
        "career_switch", "composite_index", "growth_coeff",
        "value_added", "ai_resistance",
    ]
    result = targets[[c for c in display_cols if c in targets.columns]]

    print(f"From: {from_occupation} (composite={source_row['composite_index']:.2f}, "
          f"ai_resistance={source_row['ai_resistance']:.1f})")
    print(f"Suggested transitions ({top_n}):\n")
    return result

# Example: transition_path("收银员", target_category="TECH", country="CN")


## Usage Examples

```python
# Find high-paying, AI-resistant tech jobs in China
find_jobs(country='CN', category='TECH', min_ai_resistance=7, min_salary=7)

# Compare data scientist across countries
compare('数据科学家')

# Top 15 by cost_performance in the US
top_n(metric='cost_performance', n=15, country='US')

# Country overview for Japan
country_overview('JP')

# Career transition from cashier to tech
transition_path('收银员', target_category='TECH', country='CN')
```